# Introduction to `pymovements`

- Processing eye movement data
- Open-source python package
- [Code](https://github.com/pymovements/pymovements)
- [Documentation](https://pymovements.readthedocs.io/en/latest/)
- [User Guide](https://pymovements.readthedocs.io/en/latest/user-guide/index.html)
- [Tutorials](https://pymovements.readthedocs.io/en/latest/tutorials/index.html)


## Navigation

0. [Understanding Eye-Tracking Data](./00_understanding-data.ipynb)
1. [Working with Raw Gaze Samples](./01_inspect-raw-samples.ipynb)
2. [Detecting Occumotoric Events](./02_event-detection.ipynb)
3. [Downloading Public Datasets](./03_public-datasets.ipynb)

Bonus: [Plotting Summary](./10_plotting.ipynb)

_Acknowledgement: This presentation builds on work of several `pymovements` authors._

---

# 2. Detecting Occumotoric Events

Eye-tracking data are typically segmented into discrete events such as `fixations` and `saccades`. Fixations represent periods during which the eyes remain relatively stable, enabling visual information processing. Saccades are the rapid eye movements that shift gaze between fixations. Detecting these events and computing their properties, such as fixation duration and dispersion, or saccade amplitude and peak velocity, forms the foundation for analyzing visual behavior and understanding how participants explore a stimulus.

To showcase the event detection methods, we first create a `Gaze` object and do the necessary pre-processing as described in {doc}`Working with Raw Gaze Samples <inspect-raw-samples>`.

In [ ]:
import pymovements as pm
from pymovements.gaze.experiment import Experiment
import polars as pl

%config InlineBackend.figure_format = 'svg'

In [ ]:
# Define the experimental setup.
# The screen geometry and viewing distance are required
# to convert pixel coordinates (px) into degrees of visual angle (dva).
# The sampling rate is required for velocity computation and
# for time conversion if time_unit='step' is used.
experiment = Experiment(
    screen_width_px=1280,
    screen_height_px=1024,
    screen_width_cm=38,
    screen_height_cm=30.2,
    distance_cm=68,
    origin="upper left",
    sampling_rate=250.0,
)

# Load gaze data from a CSV file and initialize a Gaze object.
# - time_column specifies the timestamp column (standardized internally to 'time')
# - pixel_columns defines the flat CSV columns that should be grouped
#   into a structured 'pixel' component (monocular: x, y)
# - the Experiment attaches screen geometry and sampling metadata
gaze = pm.gaze.from_csv(
    "./gaze-toy-example.csv",
    experiment=experiment,
    time_column="time",
    pixel_columns=["x", "y"],
)

# Convert pixel coordinates to degrees of visual angle (dva).
# Requires a valid Experiment with screen geometry and distance.
gaze.pix2deg()

# Compute velocity from the position signal.
# Velocity is derived using the sampling rate from the Experiment.
gaze.pos2vel()

## Fixations

Fixations can be detected using one of the following algorithms:

- The `I-VT` (Velocity-Threshold Identification) method classifies each sample based on its velocity. Samples with velocities below a specified threshold are labeled as fixation points. Consecutive fixation samples are then merged into fixation events. A commonly used default threshold is 20 degrees per second, though this value may vary depending on the recording setup and research question. `pymovements` implements this methods with the {py:func}`~pymovements.events.detection.ivt` function.

- The `I-DT` (Dispersion-Threshold Identification) method groups consecutive samples whose spatial dispersion remains below a predefined threshold and whose duration exceeds a minimum value. The algorithm slides a moving window over the data: if the dispersion within the window is sufficiently small, the window is classified as a fixation and is expanded until the dispersion criterion is violated. `pymovements` function: {py:func}`~pymovements.events.detection.idt`.

In [ ]:
# Detect fixations using the I-VT (velocity threshold) algorithm.
# Detected events are stored under the name 'fixation_ivt'.
gaze.detect("ivt", name="fixation_ivt")

# Inspect the first few detected events.
# Events are stored in a structured event table.
gaze.events.frame.head(5)

### Plotting IVT Fixation Events

1. Calculate the locations of the detected fixation events.
2. Plot the scanpath of the gaze data.

In [ ]:
# Compute fixation locations using pixel coordinates
gaze.compute_event_properties(("location", {"position_column": "pixel"}))

In [ ]:
pm.plotting.scanpathplot(gaze, position_column="location", event_name="fixation_ivt")

## Event Data Quality Measures: Dispersion Measures

Measures for fixation precision:

- https://pymovements.readthedocs.io/en/latest/reference/api/pymovements.measure.samples.std_rms.html#pymovements.measure.samples.std_rms
  - TODO: short description
- https://pymovements.readthedocs.io/en/latest/reference/api/pymovements.measure.samples.rms_s2s.html
  - TODO: short description
<!-- - https://pymovements.readthedocs.io/en/latest/reference/api/pymovements.measure.samples.bcea.html -->


Taking these measures using the position column (in degrees of visual angle ($[\mathrm{dva}]$)):

In [ ]:
gaze.compute_event_properties("std_rms")
gaze.compute_event_properties("rms_s2s")
# gaze.compute_event_properties("bcea")

In [ ]:
gaze.events.frame.head(5)

In [ ]:
gaze.events.frame["std_rms"].mean(), gaze.events.frame["rms_s2s"].mean()

0.059 $[\mathrm{dva}]$ and 0.025 $[\mathrm{dva}]$ in this context means...

- a small values means precise fixations,
- large ones

-> the spread of a fixation.

## Saccades

Saccades are rapid eye movements that shift the point of fixation from one location to another. In `pymovements`, saccades (including microsaccades) can be detected from the velocity signal using the {py:func}`~pymovements.events.detection.microsaccades` function. This method implements a **noise-adaptive velocity threshold**. Instead of using a fixed velocity cutoff, the threshold is scaled relative to the noise level of the velocity signal. This makes the detection procedure more robust across recordings with different noise characteristics.

Two key parameters are necessary for the identification of saccades:

- **`threshold_factor`** controls how strict the velocity threshold is (default: `6`). Higher values detect fewer saccades (more conservative), lower values detect more (more sensitive).

- **`minimum_duration`** defines the minimum length of a velocity peak to be considered a saccade (default: `6` samples). Shorter events are treated as noise and ignored.

In [ ]:
gaze.detect("microsaccades", minimum_duration=6, threshold_factor=6)

# sort by onset in order to see both fixations and saccades
gaze.events.frame = gaze.events.frame.sort("onset")

gaze.events.frame.head(10)

For more information on the algorithms and additional parameters, see the following tutorials: {doc}`Handling Gaze Events <../tutorials/event-handling>` and {doc}`Plotting Gaze Data <../tutorials/plotting>`.

### Saccadic Main Sequence

To visualise these we can plot something called saccadic main sequence.

The saccadic main sequence describes the characteristic relationship between a saccade’s amplitude and its peak velocity: larger saccades tend to be faster, following a nonlinear, saturating curve. It is commonly used to validate saccade detection and assess data quality, since deviations from the expected pattern can indicate recording errors or atypical oculomotor behavior.

In [ ]:
# compute amplitude and peak velocity of the detected saccades
gaze.compute_event_properties(["amplitude", "peak_velocity"])

In [ ]:
fig, axs = pm.plotting.main_sequence_plot(gaze.events, fit=False)

## Event ratios

In [ ]:
saccade_ratio = gaze.measure_events_ratio(
    name="saccade", time_column="time"
)  # , sampling_rate=250)

In [ ]:
pl.select(saccade_ratio)

In [ ]:
fixation_ivt_ratio = gaze.measure_events_ratio(name="fixation_ivt", time_column="time")

In [ ]:
pl.select(fixation_ivt_ratio)